# readme-drift — Interactive Demo

This notebook walks through each layer of the library with live, editable examples.

**Sections**
1. [Setup](#1-setup)
2. [AST diff — Python symbol changes](#2-ast-diff)
3. [Config diff — JSON / YAML / TOML changes](#3-config-diff)
4. [README scanner — finding symbol references](#4-readme-scanner)
5. [End-to-end: manual pipeline](#5-end-to-end)
6. [Git utilities](#6-git)
7. [run_check on the real repo](#7-run_check)
8. [Formatted report output](#8-report)
9. [CLI and pyproject.toml configuration](#9-cli)

---
## 1 · Setup

Select the venv kernel: **Python 3 (`.venv`)** — or run the notebook with  
`../.venv/bin/jupyter notebook notebooks/demo.ipynb` from the repo root.

In [ ]:
import sys
import pathlib

repo_root = pathlib.Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("repo root:", repo_root)
print("Python:   ", sys.executable)

---
## 2 · AST diff

`diff_apis(old_source, new_source)` parses two versions of a `.py` file with Python's `ast` module  
and returns a list of `SymbolChange` objects for every public symbol that was added, removed,  
or had its signature changed.

### Under the hood: the Python `ast` module

`ast_diff.py` is built directly on the standard-library `ast` module.  
Before looking at what readme-drift does with it, here's how the module itself works.

In [ ]:
import ast

source = """\
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

class Greeter:
    def say(self, msg, *, loud=False):
        ...
"""

# ast.parse returns a Module node — the root of the AST
tree = ast.parse(source)
print(type(tree))            # <class 'ast.Module'>
print(type(tree.body[0]))    # first statement: FunctionDef
print(type(tree.body[1]))    # second statement: ClassDef

In [ ]:
# ast.dump renders the full tree as a string — useful for exploration
print(ast.dump(tree, indent=2))

In [ ]:
# ast.iter_child_nodes gives direct children of a node (one level deep)
# extract_public_api uses this to visit only top-level definitions
for node in ast.iter_child_nodes(tree):
    print(type(node).__name__, "→", getattr(node, "name", ""))

In [ ]:
# FunctionDef carries name, args, and more
func_node = tree.body[0]   # greet(name, greeting="Hello")
assert isinstance(func_node, ast.FunctionDef)

print("name          :", func_node.name)
print("args.args     :", [a.arg for a in func_node.args.args])
print("args.defaults :", [ast.unparse(d) for d in func_node.args.defaults])
print("args.kwonlyargs:", [a.arg for a in func_node.args.kwonlyargs])

#### `posonlyargs` vs `args`, and the `defaults` alignment problem

Python has three kinds of positional parameters:

```
def f(a, b=1, /, c=2, d=3, *, e=4):
       ──────    ──────────   ────
       posonlyargs   args    kwonlyargs
```

- **`posonlyargs`** — parameters before `/`. Callers *cannot* pass them by keyword.
- **`args`** — regular parameters after `/`. Can be passed positionally or by keyword.
- **`kwonlyargs`** — after `*`. Keyword-only; have their own `kw_defaults` list (one entry per kwonlyarg, `None` if no default).

The tricky part is **`defaults`**: it is a *single* right-aligned list that covers **both** `posonlyargs` and `args` together.  
If the combined list `[*posonlyargs, *args]` has `N` entries and `defaults` has `D` entries, then only the **last `D` entries** have defaults — the first `N - D` do not.

In [ ]:
import ast

# a and b are positional-only (before /); c and d are regular; e is keyword-only
node = ast.parse("def f(a, b=1, /, c=2, d=3, *, e=4): ...").body[0]
fa = node.args

print("posonlyargs :", [a.arg for a in fa.posonlyargs])   # [a, b]
print("args        :", [a.arg for a in fa.args])           # [c, d]
print("kwonlyargs  :", [a.arg for a in fa.kwonlyargs])     # [e]
print()
print("defaults    :", [ast.unparse(d) for d in fa.defaults])    # [1, 2, 3]  ← shared!
print("kw_defaults :", [ast.unparse(d) for d in fa.kw_defaults]) # [4]        ← separate

In [ ]:
# Mapping defaults back to params: right-align defaults over [*posonlyargs, *args]
combined = [a.arg for a in fa.posonlyargs] + [a.arg for a in fa.args]
N = len(combined)   # 4
D = len(fa.defaults)  # 3  → first N-D=1 params have no default

print(f"combined ({N}): {combined}")
print(f"defaults ({D}): {[ast.unparse(d) for d in fa.defaults]}")
print(f"first {N - D} param(s) have no default\n")

for i, param in enumerate(combined):
    default_index = i - (N - D)   # negative → no default
    if default_index >= 0:
        print(f"  {param} = {ast.unparse(fa.defaults[default_index])}")
    else:
        print(f"  {param}  (no default)")

In [ ]:
# ClassDef has a name and a body list of its method FunctionDef nodes
class_node = tree.body[1]   # Greeter
assert isinstance(class_node, ast.ClassDef)

print("class name:", class_node.name)
for item in class_node.body:
    if isinstance(item, ast.FunctionDef):
        method_args = [a.arg for a in item.args.args if a.arg != "self"]
        kwonly     = [a.arg for a in item.args.kwonlyargs]
        print(f"  method: {item.name}  args={method_args}  kwonly={kwonly}")

In [ ]:
# ast.unparse reconstructs source text from any node — used by _format_signature
# to render default values like "Hello" back into the signature string
default_node = func_node.args.defaults[0]   # the AST node for "Hello"
print("default AST node :", ast.dump(default_node))
print("ast.unparse      :", ast.unparse(default_node))

# Putting it all together: _format_signature from ast_diff.py
from readme_drift.ast_diff import _format_signature
print("_format_signature(greet) :", _format_signature(func_node))

say_node = class_node.body[0]   # say(self, msg, *, loud=False)
print("_format_signature(say)   :", _format_signature(say_node))

### 2a · Function removed

Rename detection was removed: the "same params = rename" heuristic was too unreliable — zero-parameter functions always matched each other, and common parameter names (e.g. `host, port`) caused false renames between unrelated functions.

A renamed function is now reported as the old name being **removed** and the new name being **added**. Since `ADDED` never produces a README finding, only the removal is surfaced — which is exactly right: the README still references the old name and needs updating.

In [ ]:
from readme_drift.ast_diff import diff_apis, extract_public_api

old = """
def connect(host, port):
    ...
"""

new = """
def connect_url(host, port):   # renamed — old name reported as REMOVED, new as ADDED
    ...
"""

for c in diff_apis(old, new, file="client.py"):
    print(f"{c.change_type.name:<20} {c.name}")

### 2b · Signature changed

In [ ]:
old = """
def connect(host, port):
    ...
"""

new = """
def connect(url, timeout=30):
    ...
"""

for c in diff_apis(old, new, file="client.py"):
    print(c)
    print(f"  change_type   : {c.change_type}")
    print(f"  old_signature : {c.old_signature}")
    print(f"  new_signature : {c.new_signature}")

### 2c · Class removed + method signature changed

In [ ]:
old = """
class Client:
    def connect(self, host, port):
        ...
    def disconnect(self):
        ...

class LegacyClient:
    def send(self, data):
        ...
"""

new = """
class Client:
    def connect(self, url):   # signature changed
        ...
    def disconnect(self):
        ...
# LegacyClient removed entirely
"""

for c in diff_apis(old, new, file="client.py"):
    print(c)

### 2d · Private symbols are ignored by design

In [ ]:
old = """
def _internal_helper(x):
    ...

def public_api(x):
    ...
"""

new = """
# _internal_helper removed — private, must NOT be flagged

def public_api(x, y):   # signature changed — MUST be flagged
    ...
"""

changes = diff_apis(old, new, file="utils.py")
print(f"{len(changes)} change(s) detected:")
for c in changes:
    print(" ", c)

### 2e · Inspect the raw PublicAPI snapshot

In [ ]:
source = """
def greet(name, greeting="Hello"):
    ...

class Greeter:
    def say(self, msg, *, loud=False):
        ...
    def _reset(self):    # private — excluded
        ...
"""

api = extract_public_api(source)
print("Functions:", api.functions)
print("Classes  :", api.classes)
print("Methods  :", api.methods)

### 2f · Under the hood: `_diff_signatures`

`_diff_signatures(old, new, file)` is the core comparison function called by `diff_apis` for both top-level functions and class methods. It receives two `dict[str, str]` maps of `name → signature_string` and returns a flat list of `SymbolChange` objects covering three cases:

1. **Removed** — names in `old` but not `new` → `REMOVED`
2. **Added** — names in `new` but not `old` → `ADDED`
3. **Signature changed** — names present in both whose signature strings differ → `SIGNATURE_CHANGED`

Names with identical signatures in both versions produce no output.

Note: rename detection was intentionally removed. A rename appears as `REMOVED` (old name) + `ADDED` (new name). Since `ADDED` never produces a README finding, the net result is correct: the README is flagged for referencing the old name.

In [ ]:
from readme_drift.ast_diff import _diff_signatures

# Simulate the name→signature maps that extract_public_api produces
old = {
    "connect":      "connect(host, port)",          # removed (renamed to connect_url)
    "disconnect":   "disconnect()",                 # unchanged — should NOT appear
    "send":         "send(data)",                   # signature will change
    "legacy_ping":  "legacy_ping()",                # removed outright
}
new = {
    "connect_url":  "connect_url(host, port)",      # new name, reported as ADDED
    "disconnect":   "disconnect()",                 # identical → no change
    "send":         "send(data, encoding='utf-8')", # signature changed
    "health_check": "health_check()",               # brand-new → ADDED
}

changes = _diff_signatures(old, new, file="client.py")
for c in changes:
    print(f"{c.change_type.name:<20} {c.name}")
    if c.old_signature:
        print(f"  old: {c.old_signature}")
    if c.new_signature:
        print(f"  new: {c.new_signature}")

---
## 3 · Config diff

`diff_config(old_source, new_source, file)` compares two versions of a JSON, YAML, or TOML file  
and emits `SymbolChange` entries for key segments that were added or removed.

### 3a · JSON — npm script key removed

In [ ]:
from readme_drift.config_diff import diff_config

old_json = '{"scripts": {"build": "tsc", "test": "jest", "lint": "eslint ."}}'
new_json = '{"scripts": {"test": "jest", "lint": "eslint ."}}'

for c in diff_config(old_json, new_json, file="package.json"):
    print(c)

### 3b · YAML — CI job renamed

In [ ]:
old_yaml = """
jobs:
  build:
    runs-on: ubuntu-latest
  test:
    runs-on: ubuntu-latest
"""

new_yaml = """
jobs:
  compile:
    runs-on: ubuntu-latest
  test:
    runs-on: ubuntu-latest
"""

for c in diff_config(old_yaml, new_yaml, file="ci.yml"):
    print(c)

### 3c · TOML — tool section removed

In [ ]:
old_toml = """
[tool.black]
line-length = 88

[tool.ruff]
line-length = 88
"""

new_toml = """
[tool.ruff]
line-length = 88
"""

for c in diff_config(old_toml, new_toml, file="pyproject.toml"):
    print(c)

### 3d · Under the hood: extractors and the `KeyExtractor` protocol

`diff_config` dispatches to a format-specific extractor based on the file extension.
Each extractor implements `KeyExtractor` — a `@runtime_checkable` protocol with a single
method: `extract(source: str) -> dict[str, str]` that returns a flat dot-notation
key-path → value map. `CONFIG_SUFFIXES` lists every extension that triggers a config diff.

In [ ]:
from readme_drift.config_diff import CONFIG_SUFFIXES

# File extensions that trigger a config diff
print("Tracked config extensions:", sorted(CONFIG_SUFFIXES))

In [ ]:
from readme_drift.config_diff import JsonExtractor, TomlExtractor, YamlExtractor

# Each extractor parses its format into a flat key-path → value dict.
# diff_config calls extractor.extract() on both old and new source, then diffs the two dicts.

json_ex = JsonExtractor()
print("JSON flat keys:")
for k, v in json_ex.extract('{"scripts": {"build": "tsc", "test": "jest"}}').items():
    print(f"  {k!r}: {v!r}")
print()

toml_ex = TomlExtractor()
print("TOML flat keys:")
for k, v in toml_ex.extract("[tool.black]\nline-length = 88\n[tool.ruff]\nline-length = 88\n").items():
    print(f"  {k!r}: {v!r}")
print()

yaml_ex = YamlExtractor()
print("YAML flat keys:")
for k, v in yaml_ex.extract("jobs:\n  build:\n    runs-on: ubuntu-latest\n  test:\n    runs-on: ubuntu-latest\n").items():
    print(f"  {k!r}: {v!r}")

#### Implementing a custom `KeyExtractor`

Any class with `extract(source: str) -> dict[str, str]` satisfies the protocol.
Register it in `config_diff._EXTRACTORS` keyed by file extension to make `diff_config`
dispatch to it for that format — no other module needs to change.

In [ ]:
from readme_drift import config_diff
from readme_drift.config_diff import KeyExtractor

class DotenvExtractor:
    """Treat each KEY= line in a .env file as a tracked symbol."""

    def extract(self, source: str) -> dict[str, str]:
        result = {}
        for line in source.splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, _, value = line.partition("=")
                result[key.strip()] = value.strip()
        return result

# @runtime_checkable lets isinstance() verify protocol satisfaction at runtime
print("Satisfies KeyExtractor:", isinstance(DotenvExtractor(), KeyExtractor))

# Register so diff_config dispatches to it for .env files
config_diff._EXTRACTORS[".env"] = DotenvExtractor()

old_env = "DATABASE_URL=postgres://localhost/dev\nAPI_KEY=abc123\nDEBUG=true"
new_env = "DATABASE_URL=postgres://localhost/prod\nDEBUG=true"

for c in config_diff.diff_config(old_env, new_env, file="app.env"):
    print(c)

# Restore
del config_diff._EXTRACTORS[".env"]

---
## 4 · README scanner

`find_symbol_in_readme` searches a README file for backtick references and word-boundary occurrences  
of a symbol name. `scan_readme_for_symbols` does the same for a batch of symbols.

In [ ]:
import tempfile
import pathlib
from readme_drift.scanner import find_symbol_in_readme, scan_readme_for_symbols

README_TEXT = """
# My Library

Call `connect(host, port)` to open a connection.

The build can be run using `npm run build` to compile the project.

Use the `Client` class directly if you need fine-grained control.

Internally `_helper` is used but not part of the public API.
"""

tmp = pathlib.Path(tempfile.mktemp(suffix=".md"))
tmp.write_text(README_TEXT)

for symbol in ["connect", "build", "Client", "_helper", "nonexistent", "run"]:
    matches = find_symbol_in_readme(tmp, symbol)
    print(matches)

In [ ]:
# Batch scan — returns only symbols that were actually found
results = scan_readme_for_symbols(tmp, ["connect", "build", "Client", "nonexistent"])
print("Symbols found:", list(results.keys()))

---
## 5 · End-to-end: manual pipeline

Wire all three layers together without touching git — useful for unit-testing scenarios  
or understanding how `drift_checker.run_check` works internally.

In [ ]:
import tempfile
import pathlib
from readme_drift.ast_diff import diff_apis
from readme_drift.scanner import scan_readme_for_symbols
from readme_drift.drift_checker import _symbols_from_changes
from readme_drift.models import DriftCheckResult, StalenessFinding, ChangeType
from readme_drift.report import format_report

README_TEXT = """
# Acme SDK

Use `Client.connect(host, port)` to open a connection to the server.

Call `build` in your CI pipeline via `npm run build`.

The `send_payload` function handles serialisation.
"""

old_py = """
class Client:
    def connect(self, host, port):
        ...

def send_payload(data):
    ...
"""

new_py = """
class Client:
    def connect(self, url):               # signature changed
        ...

def send_payload(data, encoding="utf-8"): # signature changed
    ...
"""

# 1 — diff the code
changes = diff_apis(old_py, new_py, file="sdk/client.py")

# 2 — scan the README for the changed symbol names
tmp_readme = pathlib.Path(tempfile.mktemp(suffix=".md"))
tmp_readme.write_text(README_TEXT)
readme_matches = scan_readme_for_symbols(tmp_readme, _symbols_from_changes(changes))

# 3 — build findings (skip ADDED symbols — a README that already documents new API is fine)
findings = [
    StalenessFinding(change=c, readme_matches=readme_matches[c.name])
    for c in changes
    if c.change_type != ChangeType.ADDED and c.name in readme_matches
]

print(findings)

result = DriftCheckResult(findings=findings, readme_paths=[tmp_readme])
print(format_report(result))

---
## 6 · Git utilities

The `git` module handles all subprocess and filesystem operations.
These functions are what `run_check` calls internally — useful to understand
when integrating readme-drift into custom tooling or debugging unexpected behaviour.

In [ ]:
from readme_drift.git import get_repo_root, validate_repo_root
import tempfile, pathlib

# get_repo_root() walks up from cwd until it finds a .git directory
detected = get_repo_root()
print("Detected repo root:", detected)

# validate_repo_root() confirms a path is a valid git repo, raises ValueError otherwise
validated = validate_repo_root(repo_root)
print("Validated root    :", validated)

# Non-git paths raise ValueError
tmp_dir = pathlib.Path(tempfile.mkdtemp())
try:
    validate_repo_root(tmp_dir)
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
from readme_drift.git import find_readmes

# find_readmes walks the repo recursively (case-insensitive name match, multiple extensions).
# Skips: .git, node_modules, venv, .venv, .tox, __pycache__, .pytest_cache, dist, build, .mypy_cache
readmes = find_readmes(repo_root)
for r in readmes:
    print(r.relative_to(repo_root))

In [ ]:
# get_diff runs `git diff --name-only <base_ref>` and returns a GitDiffResult.
# base_ref="HEAD" compares the working tree to the last commit.
diff = get_diff(base_ref="HEAD", repo_root=repo_root)
print("Changed Python files :", diff.changed_py_files)
print("Changed config files :", diff.changed_config_files)

In [ ]:
from readme_drift.git import get_diff

# staged=True uses `git diff --cached` — compares the index to HEAD.
# This is the mode the pre-commit hook uses (only what has been `git add`-ed is checked).
diff_staged = get_diff(staged=True, repo_root=repo_root)
print("Staged Python files :", diff_staged.changed_py_files)
print("Staged config files :", diff_staged.changed_config_files)

In [ ]:
# read_old_content fetches a file at a given git ref via `git show <ref>:<path>`.
# Returns empty string if the file didn't exist at that ref (e.g. newly added files).
if diff.changed_py_files:
    py_file = diff.changed_py_files[0]
    old_src = read_old_content(py_file, repo_root, "HEAD")
    print(f"Old content of '{py_file}' at HEAD ({len(old_src)} chars):")
    print(old_src[:300] + "..." if len(old_src) > 300 else old_src)
else:
    print("No changed Python files — edit a .py file to try this.")

In [ ]:
from readme_drift.git import read_old_content, read_new_content

# read_new_content reads from disk when staged=False, or from the git index when staged=True.
if diff.changed_py_files:
    py_file = diff.changed_py_files[0]
    new_src = read_new_content(py_file, repo_root, repo_root.resolve(), staged=False)
    print(f"New content of '{py_file}' from disk ({len(new_src)} chars):")
    print(new_src[:300] + "..." if len(new_src) > 300 else new_src)
else:
    print("No changed Python files — edit a .py file (without staging) to try this.")

---
## 7 · run_check on the real repo

`run_check` is the public entry point used by the CLI and pre-commit hook.
It shells out to `git diff` to get the real changed files.

In [ ]:
from readme_drift.drift_checker import run_check
from readme_drift.report import format_report

# Diff the working tree against HEAD (same as running `readme-drift` with no args)
result = run_check(base_ref="HEAD", repo_root=repo_root)
print(format_report(result))

In [ ]:
# Staged-only mode — same as what the pre-commit hook uses
result_staged = run_check(staged=True, repo_root=repo_root)
print(format_report(result_staged))

---
## 8 · Formatted report output

Build `DriftCheckResult` / `StalenessFinding` objects by hand to explore what the formatted output looks like.

In [ ]:
import pathlib
from readme_drift.models import (
    ChangeType, SymbolChange, ReadmeMatch, StalenessFinding, DriftCheckResult,
)
from readme_drift.report import format_report

result = DriftCheckResult(
    findings=[
        StalenessFinding(
            change=SymbolChange(
                name="Client.connect",
                change_type=ChangeType.SIGNATURE_CHANGED,
                old_signature="Client.connect(host, port)",
                new_signature="Client.connect(url)",
                file="src/client.py",
            ),
            readme_matches=[
                ReadmeMatch(
                    symbol="Client.connect",
                    line_number=42,
                    line_text="Call `Client.connect(host, port)` to open a connection.",
                    matched_text="`Client.connect(host, port)`",
                    readme_path=pathlib.Path("README.md"),
                )
            ],
        ),
        StalenessFinding(
            change=SymbolChange(
                name="build",
                change_type=ChangeType.REMOVED,
                file="package.json",
            ),
            readme_matches=[
                ReadmeMatch(
                    symbol="build",
                    line_number=18,
                    line_text="Run `npm run build` to compile.",
                    matched_text="`build`",
                    readme_path=pathlib.Path("README.md"),
                )
            ],
        ),
    ],
    readme_paths=[pathlib.Path("README.md")],
)

print(format_report(result))
print()
print("passed:", result.passed, " failed:", result.failed)

In [ ]:
# Skipped and clean variants
print(format_report(DriftCheckResult(skipped=True, skip_reason="no Python or config files changed")))
print()
print(format_report(DriftCheckResult(readme_paths=[pathlib.Path("README.md")])))

---
## 9 · CLI and `pyproject.toml` configuration

`cli.py` wraps `run_check` with `argparse` and loads project-level defaults from
`[tool.readme-drift]` in `pyproject.toml`. The precedence is:
**CLI flag > pyproject.toml > built-in default**.

In [ ]:
import subprocess, sys

# Show the full CLI help
result = subprocess.run(
    [sys.executable, "-m", "readme_drift.cli", "--help"],
    capture_output=True, text=True, cwd=str(repo_root),
)
print(result.stdout)

#### `pyproject.toml` config loading

`_load_toml_config` searches for `pyproject.toml` starting from `--repo-root` (or cwd),
reads the `[tool.readme-drift]` section, and feeds those as `argparse` defaults.
CLI flags always override the file. A minimal example:

```toml
[tool.readme-drift]
base-ref = "origin/main"
warn-only = false
include-private = false
exclude = ["generated/", "tests/"]
plain-text-search = true
```

In [ ]:
from readme_drift.cli import _load_toml_config

# _load_toml_config searches for pyproject.toml starting from repo_root (or cwd).
# Returns the [tool.readme-drift] dict — empty if the section is absent.
cfg = _load_toml_config(repo_root)
print("Current [tool.readme-drift] config:", cfg)
print()

# Effective defaults when a key is absent from the TOML:
print("Effective defaults:")
print(f"  base-ref          = {cfg.get('base-ref', 'HEAD')!r}")
print(f"  warn-only         = {cfg.get('warn-only', False)}")
print(f"  include-private   = {cfg.get('include-private', False)}")
print(f"  exclude           = {cfg.get('exclude', [])}")
print(f"  plain-text-search = {cfg.get('plain-text-search', True)}")

#### Invoking from Python or shell

Boolean flags use `--flag=true/false` syntax (accepts `true/false`, `1/0`, `yes/no`).

```bash
readme-drift --base-ref=origin/main
readme-drift --staged=true                     # pre-commit mode
readme-drift --warn-only=true                  # never fail CI
readme-drift --include-private=true
readme-drift --plain-text-search=false         # backtick spans only
readme-drift --exclude=generated/ --exclude=tests/
```

In [ ]:
import subprocess, sys

# Invoke readme-drift programmatically — same as running it in the shell.
# --warn-only=true ensures exit 0 even if findings exist, so the notebook doesn't error.
result = subprocess.run(
    [
        sys.executable, "-m", "readme_drift.cli",
        "--warn-only=true",
        f"--repo-root={repo_root}",
        "--base-ref=HEAD",
    ],
    capture_output=True, text=True, cwd=str(repo_root),
)
print(result.stdout or result.stderr)
print("exit code:", result.returncode)